In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv
import sys
import json

In [2]:
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [3]:
load_dotenv()
src_path = os.getenv("SRC_PATH")
data_path = os.getenv("DATA_PATH")
transaction_path = os.path.join(data_path, r'raw/train_transaction.csv/train_transaction.csv')
identity_path = os.path.join(data_path, r'raw/train_identity.csv/train_identity.csv')
train_transaction = pd.read_csv(transaction_path)
train_identity = pd.read_csv(identity_path)
df = train_transaction.merge(train_identity, on="TransactionID", how = "left")
df = df.sort_values("TransactionDT").reset_index(drop=True)

In [4]:
sys.path.append(src_path)
from features.engineering import create_engineered_features
from data.split import temporal_split

In [5]:
json_path = os.path.join(data_path, "processed", "split_info.json")
with open(json_path, 'r') as f:
    split_info = json.load(f)

train_end = split_info.get("train_end")
val_end = split_info.get("validation_end")

In [6]:
train_df, val_df, test_df = temporal_split(df, train_end, val_end)
y_datasets = {}
map_dfs = {"train": train_df, "val": val_df, "test": test_df}

for name, df in map_dfs.items():
    y_datasets[f'y_{name}'] = df['isFraud']

In [7]:
def add_amount_features(df):
    df = df.copy()

    df["transaction_amt_log"] = np.log1p(df["TransactionAmt"])

    df["amount_decimal"] = (df["TransactionAmt"] % 1)

    return df

In [8]:
def create_d_time_features(df):
    df = df.copy()

    df["hour_sin"] = np.sin(2 * np.pi * df["transaction_hour"] / 24)

    df["hour_cos"] = np.cos(2 * np.pi * df["transaction_hour"] / 24)

    return df

In [18]:
def add_email_features(df):
    df = df.copy()

    for col in ["P_emaildomain", "R_emaildomain"]:
        df['col}_is_missing'] = df[col].isna().astype(int)

        df[f"{col}_provider"] = df[col].fillna("missing").str.split(".").str[0]

    return df

In [19]:
def create_d_features(df):
    df = df.copy()

    df = create_engineered_features(df)

    df = add_amount_features(df)

    df = create_d_time_features(df)

    df = add_email_features(df)

    return df

In [22]:
d_features_dfs = {}

d_features_dfs.clear()
for name, df in map_dfs.items():
    d_features_dfs[f'{name}'] = create_d_features(df)

In [25]:
assert list(d_features_dfs['train'].columns) == list(d_features_dfs['val'].columns)
assert list(d_features_dfs['train']) == list(d_features_dfs['test'].columns)